# Read source and make chunks

Goal: turn selected repository files into small searchable pieces while keeping each file path, symbol, and line range. Nothing in the selected repository is executed.

This tutorial is standalone. On a new computer, first run the setup command in the README and select QAChat's `.venv` kernel.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Python environment: {sys.executable}")

## 1. Choose and read repository files

The scanner first selects safe file names. `read_repository_files` then reads only that selection as UTF-8 text. Start with QAChat's Python package.

In [ ]:
REPOSITORY = root
INCLUDE_PATHS = ["qachat"]
FILE_TYPES = [".py"]

In [ ]:
from qachat.repository import scan_repository
from qachat.source import read_repository_files

report = scan_repository(
    REPOSITORY,
    extensions=FILE_TYPES,
    include_paths=INCLUDE_PATHS,
)
source_files = read_repository_files(report)

for source_file in source_files:
    print(f"{source_file.path}: {len(source_file.text)} characters")

## 2. Find natural source units

A natural unit is a complete Python function or class, a Markdown section, or a complete generic text file. Python is parsed as text with `ast`; it is not imported.

In [ ]:
from qachat.source import extract_source_units

units = extract_source_units(source_files)

for unit in units[:20]:
    location = f"{unit.path}:{unit.start_line}-{unit.end_line}"
    print(f"{location:<35} {unit.kind:<25} {unit.symbol}")

print(f"Total natural units: {len(units)}")

## 3. Make bounded chunks

Real indexing uses the embedding model's tokenizer. Here a visible word counter makes the splitting behavior easy to inspect. A unit stays whole when it fits; only an oversized unit is split.

In [ ]:
def count_words(text):
    return len(text.split())


TARGET_WORDS = 120
MAX_WORDS = 160
OVERLAP_WORDS = 20

In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=TARGET_WORDS,
    max_tokens=MAX_WORDS,
    overlap_tokens=OVERLAP_WORDS,
)

for chunk in chunks[:20]:
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    size = count_words(chunk.text)
    print(f"{location:<35} {size:>3} words  {chunk.symbol}")

print(f"Total chunks: {len(chunks)}")

## Result

`chunks` now contains source text plus exact provenance. It is still ordinary in-memory data: no embedding model, index, LLM, or generated files were used.